# TCM-Hermes v5 — 全功能 Colab 演示

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pariskang/TCM-Hermes/blob/claude/project-code-review-4jkbyh/notebooks/TCM_Hermes_Colab.ipynb)

> **模型自主治理的古籍规则生成系统** — 证据回源 + 对抗式质疑 + 一致性裁决 +
> 自动修复 + 发布分级；风险控制门控（分布无关有限样本保证）；十八反/十九畏
> 安全筛查；疾病多智能体发现 + ECharts 可视化；MCP 接入；litellm 多厂商后端
> （含 **Azure OpenAI** 与 **MiniMax**）。

本 notebook 依次演示**所有对外能力**。大多数单元用仓库自带产物（Gold 规则 /
Skills / 金标准）即可运行，**无需下载 68MB 语料**；最后的「完整复现」单元可选
地下载真实语料跑全管线。

运行顺序：**从上到下**。菜单 `代码执行程序 → 全部运行` 即可。

## 1. 安装
克隆仓库并以零强制依赖方式安装（`py7zr` 仅用于可选的 7z 语料解压）。

In [ ]:
import os, sys, subprocess
BRANCH = 'claude/project-code-review-4jkbyh'
if not os.path.exists('TCM-Hermes'):
    !git clone --depth 1 -b {BRANCH} https://github.com/pariskang/TCM-Hermes.git
%cd TCM-Hermes
!pip -q install -e '.[dev]'
import hermes; print('TCM-Hermes', hermes.__version__, 'installed ✓')

### 便捷封装
`hermes` 的每个能力都是纯函数（JSON 入/出）。这里用一个小工具把 JSON 结果
美观打印。CLI 等价命令写在每节标题下。

In [ ]:
import json
from hermes.config import HermesConfig
CFG = HermesConfig(root='.')

def show(obj, limit=2000):
    s = json.dumps(obj, ensure_ascii=False, indent=2)
    print(s if len(s) <= limit else s[:limit] + ' …(截断)')

## 2. 系统状态
`python3 -m hermes status`

In [ ]:
from hermes.integrations import tools  # noqa
import subprocess
print(subprocess.run([sys.executable,'-m','hermes','status'],
                     capture_output=True, text=True).stdout)

## 3. Skill RAG 问答
从已编译的 Hermes Skills（方剂 + 疾病）作答，输出发布级别 / 一致性分 / 支持
规则 / 原文证据 / 变体 / 冲突 / 安全声明。

`python3 -m hermes ask "…"`

In [ ]:
from hermes.rag.skill_rag import SkillRAGAgent
ans = SkillRAGAgent(CFG).ask('汗出惡風，脈浮緩，古籍中有哪些方證依據？')
show({k: ans[k] for k in ('selected_skill','release_level','consensus_score',
                          'abstracted_claim') if k in ans})

## 4. 方药溯源
某方剂的最早出处、历代时间线、加减方与证据链。

`python3 -m hermes lineage 桂枝湯 --brief`

In [ ]:
from hermes.lineage.formula_lineage import FormulaLineageAgent
show(FormulaLineageAgent(CFG).trace('桂枝湯'), limit=1500)

## 5. 处方 → 经典方匹配 + 安全筛查
药物集合 Jaccard + 包含度匹配；**输出附十八反/十九畏/毒性/妊娠禁忌筛查**。
支持简体输入。

`python3 -m hermes match-prescription "桂枝,白芍,炙甘草,生姜,大枣"`

In [ ]:
from hermes.lineage.prescription import PrescriptionMatcherAgent
res = PrescriptionMatcherAgent(CFG).match(['桂枝','白芍','炙甘草','生姜','大枣'])
print('top match:', res['matches'][0]['formula'], res['matches'][0]['similarity'])
print('safety risk:', res['safety']['risk_level'])

### 配伍禁忌演示（十八反）
「甘草 × 甘遂」「附子(乌头类) × 半夏」应被红旗标出。

In [ ]:
from hermes.knowledge.incompatibility import check_safety
show(check_safety(['甘草','甘遂','半夏','附子']), limit=1600)

## 6. 医师工作台
方证匹配（证据链 + 禁忌提醒 + 药物安全 + 免责声明）与经典方鉴别。

`python3 -m hermes physician match --text "…"` / `physician differentiate`

In [ ]:
from hermes.workbench.physician import DoctorAssistantAgent
doc = DoctorAssistantAgent(CFG)
m = doc.match_pattern(free_text='恶寒发热，无汗，身疼痛，脉浮紧')
for c in m['candidates'][:3]:
    print(c['formula'], c['match_score'], '| safety:',
          (c.get('herb_safety') or {}).get('risk_level'))

In [ ]:
show(doc.differentiate('桂枝湯','麻黃湯'), limit=1200)

## 7. 患者教育（安全门控）
解释中医术语；**拒绝诊断/处方/剂量请求**，红旗症状转急诊提示。

`python3 -m hermes patient explain --text "…"`

In [ ]:
from hermes.workbench.patient import PatientEducationAgent
pt = PatientEducationAgent(CFG)
print('▶ 术语解释:'); show(pt.explain('医生说我营卫不和是什么意思？'), 900)
print('\n▶ 处方请求（应被拒绝）:'); show(pt.explain('请给我开桂枝汤的剂量'), 900)

## 8. 科研工作台
统计与主题挖掘（条文 / 实体频次 / 共现 / 研究假设，输出证据链）。

`python3 -m hermes research stats` / `research mine --topic 胸痹`

In [ ]:
from hermes.workbench.researcher import ResearchWorkbench
rw = ResearchWorkbench(CFG)
stats = rw.corpus_statistics()
print('rules analyzed:', stats.get('rules_analyzed'))
top = stats.get('frequencies',{}).get('formula',[])[:5]
print('top formulas:', [(t['term'], t['count']) for t in top])

## 9. 金标准评测基准 🔬
55 条宋本伤寒论/金匮人工标注 → 抽取 P/R/F1 + 条件质量 + 门控校准。
**任何抽取器/审核改动的回归判据。**

`python3 -m hermes benchmark`

In [ ]:
from hermes.metrics.benchmark import GoldBenchmark
res = GoldBenchmark(CFG).run()
show({'micro': res['micro'],
      'gate_calibration': {k: res['gate_calibration'][k]
         for k in ('gold_precision','released_precision')}})

## 10. 风险控制门控校准 🔬🔬
把手工阈值（0.93/0.85/0.75）升级为**分布无关有限样本保证**：RCPS + 精确
Clopper-Pearson 二项上界（精度 + 召回双向），并输出 ECE / reliability /
选择性风险。详见 `docs/CALIBRATION.md`。

`python3 -m hermes calibrate`

In [ ]:
from hermes.metrics.calibration import run_calibration
payload = run_calibration(CFG)
cal = payload['calibration']
print('ECE:', cal['ece']['ece'], '| MCE:', cal['ece']['mce'])
print('calibrated τ:', cal['calibrated_thresholds'])
print('\nfixed vs risk-controlled Gold:')
g_fixed = payload['fixed_threshold_baseline']['gold']
g_cal = next(t for t in cal['tiers'] if t['tier']=='gold')
print(f"  fixed 0.93 → n={g_fixed['n_released']} errUCB={g_fixed['error_upper_bound']}")
print(f"  risk-ctrl {g_cal['threshold']} → n={g_cal['n_released']} "
      f"errUCB={g_cal['error_upper_bound']} recall≥{g_cal['empirical_recall']}")

## 11. 疾病多智能体发现 + ECharts 可视化
现代疾病 → 古籍多层检索 → 候选自治审核 → 五层本体 → 药物共现网络 → 报告 →
Skill → 9 种交互式图表（PNG/SVG/JSON 导出）。

`python3 -m hermes disease run --disease 银屑病` / `disease-viz --disease 温病`

In [ ]:
from hermes.disease.pipeline import DiseaseHermesPipeline
summary = DiseaseHermesPipeline(CFG).run('银屑病', use_sample=True)
show({k: summary[k] for k in ('display_name','candidates','by_level',
                              'by_subtype') if k in summary})

### 内联渲染 ECharts 仪表盘（温病）
导出仪表盘 HTML 并直接嵌入 notebook（含 9 图 + DIY 参数面板）。

In [ ]:
from hermes.viz import VisualizationExporter, VizParams
out = VisualizationExporter(CFG).export('温病', VizParams(top_n=20))
print('charts:', out['charts'])
from pathlib import Path
html = Path(out['files'][0]).read_text(encoding='utf-8')
from IPython.display import HTML, display
display(HTML(html))

## 12. MCP 接入（13 个工具）
零依赖 stdio MCP server，把 Hermes 能力暴露给 Claude Code / Codex / 任意 MCP
客户端。这里直接列出并调用工具（provider-agnostic 纯函数层）。

`python3 -m hermes mcp`（stdio 服务器）

In [ ]:
from hermes.integrations.tools import HERMES_TOOLS, run_tool
print(f'{len(HERMES_TOOLS)} tools:')
for t in HERMES_TOOLS:
    print('  -', t['name'])
print()
out = run_tool('hermes_prescription_safety', {'herbs':'人参,五灵脂'}, CFG)
print('人参×五灵脂 →', out['risk_level'],
      [p['rule'] for p in out['incompatible_pairs']])

## 13. litellm 多厂商后端（含 Azure / MiniMax）
默认 `heuristic`（离线确定性，本 notebook 全程用它）。切到 `litellm` 后端即可
接入几乎所有大模型，并**按 agent 角色绑定不同厂商**（真正的多模型共识）。

下面的单元**仅演示配置**（不实际调用，除非你填入自己的 key）。

### Azure OpenAI

In [ ]:
# import os
# os.environ['HERMES_BACKEND'] = 'litellm'
# os.environ['HERMES_LLM_MODEL'] = 'azure/<your-deployment>'
# os.environ['HERMES_LLM_API_BASE'] = 'https://<resource>.openai.azure.com'
# os.environ['HERMES_LLM_API_KEY'] = '<azure-key>'
# os.environ['HERMES_LLM_API_VERSION'] = '2024-02-01'
# 之后照常: python3 -m hermes review --books BOOK_SHL_SONGBEN
print('Azure 配置示例见上（取消注释并填入 key）')

### MiniMax
原生 `minimax/…`（`MINIMAX_API_KEY` 自动识别），或 OpenAI 兼容端点。

In [ ]:
# os.environ['HERMES_BACKEND'] = 'litellm'
# os.environ['HERMES_LLM_MODEL'] = 'minimax/MiniMax-M2'
# os.environ['MINIMAX_API_KEY'] = '<minimax-key>'
# (可选) os.environ['MINIMAX_API_BASE'] = 'https://api.minimax.io/v1'
print('MiniMax 配置示例见上')

### 跨厂商多模型共识（按角色绑定）
例如 **Azure 裁决 + MiniMax 对抗 + OpenAI 抽取** —— 让共识来自真正独立的模型。

In [ ]:
# 抽取用 OpenAI，对抗 critic 用 MiniMax，裁决 judge 用 Azure
# os.environ['HERMES_LLM_MODEL_EXTRACTOR'] = 'gpt-4o-mini'
# os.environ['HERMES_LLM_MODEL_CRITIC']    = 'minimax/MiniMax-M2'
# os.environ['HERMES_LLM_MODEL_JUDGE']     = 'azure/<deployment>'
# os.environ['HERMES_LLM_API_BASE_JUDGE']  = 'https://<resource>.openai.azure.com'
# os.environ['HERMES_CONSENSUS_MODE'] = 'panel'   # 启用评审小组辩论
print('每个 agent 角色可绑不同厂商/模型/端点；详见 docs/LLM_BACKENDS.md')

## 14. （可选）完整复现：下载真实语料 → 全管线
下载 68MB 中醫笈成语料（扁平布局，按书内 `分類=` 元数据归类），跑
catalog → segment → 五层审核 → 主题 → 合并 → Skill → 报告。
**耗时数分钟**，需要联网。默认注释掉；取消注释即可运行。

`bash scripts/run_full_pipeline.sh`

In [ ]:
# !python3 -m hermes download --extract
# !python3 -m hermes pipeline
# 之后 search 等需要语料的命令即可用：
# !python3 -m hermes search '陽浮而陰弱' --subcategory 傷寒 --original-only
print('取消注释以运行完整复现')

---
**文档**：`docs/HERMES_V5_PROTOCOL.md`（协议）· `docs/LLM_BACKENDS.md`（多厂商
后端）· `docs/CALIBRATION.md`（风险控制门控）· `docs/DISEASE_HERMES.md`（疾病
框架）· `docs/INTEGRATIONS.md`（MCP 接入）· `docs/SAFETY.md`（安全边界）。

本系统输出为古籍知识整理，**不构成诊疗建议**。